In [88]:
import pandas as pd
import numpy as np
from datetime import date

## 1. The Weight of Experience

### Data Needed:
- Average age of each team in each wordlcup
- WC Standings per year

### Average age of each team in each worldcup

Goal Dataset: year, team, position, avg_age

In [89]:
df_players = pd.read_csv("../data/raw/players_curated.csv")

In [90]:
df_players.head()

,player_id,family_name,given_name,birth_date,male,female,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,wikipedia_link_player
0,05acd04d-bff0-58aa-8ac8-843458f740f5,A'Court,Alan,1934-09-30,True,False,False,False,False,True,1,1958,https://en.wikipedia.org/wiki/Alan_A%27Court
1,a6cb677c-c3b8-5013-b20f-7b213b198f0e,Aaronson,Brenden,2000-10-22,True,False,False,False,False,True,1,2022,https://en.wikipedia.org/wiki/Brenden_Aaronson
2,fed9938a-eeac-599e-ad8f-03d456de0420,Aarønes,Ann Kristin,1973-01-19,False,True,False,False,True,True,2,"1999, 1995",https://en.wikipedia.org/wiki/Ann_Kristin_Aar%...
3,f450d67c-2ad0-5052-9450-d559568dae61,Abadzhiev,Stefan,1934-07-03,True,False,False,False,False,True,1,1966,https://en.wikipedia.org/wiki/Stefan_Abadzhiev
4,cb2815a6-1881-5197-9b8f-fbf35d2a3c4c,Abalo,Jean-Paul,1975-06-26,True,False,False,True,False,False,1,2006,https://en.wikipedia.org/wiki/Jean-Paul_Abalo


In [91]:
df_players = df_players[df_players["male"]==True]
df_players = df_players[["family_name", "player_id", "birth_date", "goal_keeper", "defender", "midfielder", "forward", "list_tournaments"]]

In [92]:
df_players.head()

,family_name,player_id,birth_date,goal_keeper,defender,midfielder,forward,list_tournaments
0,A'Court,05acd04d-bff0-58aa-8ac8-843458f740f5,1934-09-30,False,False,False,True,1958
1,Aaronson,a6cb677c-c3b8-5013-b20f-7b213b198f0e,2000-10-22,False,False,False,True,2022
3,Abadzhiev,f450d67c-2ad0-5052-9450-d559568dae61,1934-07-03,False,False,False,True,1966
4,Abalo,cb2815a6-1881-5197-9b8f-fbf35d2a3c4c,1975-06-26,False,True,False,False,2006
6,Abanda,4ae54ba2-ab91-5924-8b8f-e1fbb43f8769,1978-08-03,False,True,False,False,1998


In [93]:
# Split players who played more than one tournament
df_players["list_tournaments"] = df_players["list_tournaments"].str.split(", ")
df_players = df_players.explode("list_tournaments")
df_players["list_tournaments"] = df_players["list_tournaments"].astype(int)

In [94]:
# Get player age at WC
df_players['birth_date'] = pd.to_datetime(df_players['birth_date'])

df_players["age"] = df_players["list_tournaments"] - df_players["birth_date"].dt.year
df_players = df_players.drop('birth_date', axis=1)


In [95]:
df_players.head()

,family_name,player_id,goal_keeper,defender,midfielder,forward,list_tournaments,age
0,A'Court,05acd04d-bff0-58aa-8ac8-843458f740f5,False,False,False,True,1958,24.0
1,Aaronson,a6cb677c-c3b8-5013-b20f-7b213b198f0e,False,False,False,True,2022,22.0
3,Abadzhiev,f450d67c-2ad0-5052-9450-d559568dae61,False,False,False,True,1966,32.0
4,Abalo,cb2815a6-1881-5197-9b8f-fbf35d2a3c4c,False,True,False,False,2006,31.0
6,Abanda,4ae54ba2-ab91-5924-8b8f-e1fbb43f8769,False,True,False,False,1998,20.0


In [96]:
# Add team to players
df_squads = pd.read_csv("../data/raw/squads_curated.csv")
df_squads.head()

,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,squad_position_name,squad_position_code
0,013eb3e7-ff79-5592-944c-ca635e888abb,2019 FIFA Women's World Cup,933a103d-d9d7-57a3-9c4a-fbc81d25a266,Argentina,ARG,0b97bb1b-8492-5982-886f-b5a49e287fe0,Correa,Vanina,1.0,goal keeper,GK
1,013eb3e7-ff79-5592-944c-ca635e888abb,2019 FIFA Women's World Cup,933a103d-d9d7-57a3-9c4a-fbc81d25a266,Argentina,ARG,f549934b-a2b3-5560-82d9-19734d6db351,Barroso,Agustina,2.0,defender,DF
2,013eb3e7-ff79-5592-944c-ca635e888abb,2019 FIFA Women's World Cup,933a103d-d9d7-57a3-9c4a-fbc81d25a266,Argentina,ARG,3de47b3b-b5b9-53c8-bf01-bc94d232d4b2,Stábile,Eliana,3.0,defender,DF
3,013eb3e7-ff79-5592-944c-ca635e888abb,2019 FIFA Women's World Cup,933a103d-d9d7-57a3-9c4a-fbc81d25a266,Argentina,ARG,af18292a-f644-55b9-b676-b0acd66a55a8,Sachs,Adriana,4.0,defender,DF
4,013eb3e7-ff79-5592-944c-ca635e888abb,2019 FIFA Women's World Cup,933a103d-d9d7-57a3-9c4a-fbc81d25a266,Argentina,ARG,a81dfc6c-2e7c-550b-a434-88d5ed21018c,Santana,Vanesa,5.0,midfielder,MF


In [97]:
df_players = df_players.merge(df_squads[["player_id", "team_name"]], on="player_id")
df_players.head()

,family_name,player_id,goal_keeper,defender,midfielder,forward,list_tournaments,age,team_name
0,A'Court,05acd04d-bff0-58aa-8ac8-843458f740f5,False,False,False,True,1958,24.0,England
1,Aaronson,a6cb677c-c3b8-5013-b20f-7b213b198f0e,False,False,False,True,2022,22.0,United States
2,Abadzhiev,f450d67c-2ad0-5052-9450-d559568dae61,False,False,False,True,1966,32.0,Bulgaria
3,Abalo,cb2815a6-1881-5197-9b8f-fbf35d2a3c4c,False,True,False,False,2006,31.0,Togo
4,Abanda,4ae54ba2-ab91-5924-8b8f-e1fbb43f8769,False,True,False,False,1998,20.0,Cameroon


In [98]:
# Transform position into one column
# Some players (7%) have more then one position in the same world cup, so it will be assigned according the this hierarchy: Goalkeeper > Defender > Midfielder > Forward
position_cols = [
    "goal_keeper",
    "defender",
    "midfielder",
    "forward"
]

multi_position_players = df_players[
    df_players[position_cols].sum(axis=1) > 1
]

multi_position_players

,family_name,player_id,goal_keeper,defender,midfielder,forward,list_tournaments,age,team_name
95,Acuña,58fc463f-7063-541e-b077-cbfb82d5aadf,False,True,True,False,2018,27.0,Argentina
96,Acuña,58fc463f-7063-541e-b077-cbfb82d5aadf,False,True,True,False,2018,27.0,Argentina
97,Acuña,58fc463f-7063-541e-b077-cbfb82d5aadf,False,True,True,False,2022,31.0,Argentina
98,Acuña,58fc463f-7063-541e-b077-cbfb82d5aadf,False,True,True,False,2022,31.0,Argentina
158,Afonin,3fc044ed-894b-5be3-b612-15e67aafbda9,False,True,True,False,1966,27.0,Soviet Union
...,...,...,...,...,...,...,...,...,...
17055,Šerić,53abd99a-3fdc-5abe-822b-e56b2ac20072,False,True,True,False,1998,19.0,Croatia
17093,Šurjak,09c2d73e-c2b1-5483-8009-940b860884b6,False,False,True,True,1974,21.0,Yugoslavia
17094,Šurjak,09c2d73e-c2b1-5483-8009-940b860884b6,False,False,True,True,1974,21.0,Yugoslavia
17095,Šurjak,09c2d73e-c2b1-5483-8009-940b860884b6,False,False,True,True,1982,29.0,Yugoslavia


In [99]:
df_players["position"] = np.select(
    [
        df_players["goal_keeper"],
        df_players["defender"],
        df_players["midfielder"],
        df_players["forward"],
    ],
    [
        "Goalkeeper",
        "Defender",
        "Midfielder",
        "Forward",
    ],
    default="Unknown"
)

df_players = df_players.drop_duplicates()

In [100]:
df_players.head()

,family_name,player_id,goal_keeper,defender,midfielder,forward,list_tournaments,age,team_name,position
0,A'Court,05acd04d-bff0-58aa-8ac8-843458f740f5,False,False,False,True,1958,24.0,England,Forward
1,Aaronson,a6cb677c-c3b8-5013-b20f-7b213b198f0e,False,False,False,True,2022,22.0,United States,Forward
2,Abadzhiev,f450d67c-2ad0-5052-9450-d559568dae61,False,False,False,True,1966,32.0,Bulgaria,Forward
3,Abalo,cb2815a6-1881-5197-9b8f-fbf35d2a3c4c,False,True,False,False,2006,31.0,Togo,Defender
4,Abanda,4ae54ba2-ab91-5924-8b8f-e1fbb43f8769,False,True,False,False,1998,20.0,Cameroon,Defender


In [101]:
df_players = df_players[["list_tournaments", "team_name", "position", "age"]]
df_players.head()

,list_tournaments,team_name,position,age
0,1958,England,Forward,24.0
1,2022,United States,Forward,22.0
2,1966,Bulgaria,Forward,32.0
3,2006,Togo,Defender,31.0
4,1998,Cameroon,Defender,20.0


In [102]:
# Add avg_age (not per position)
df_players["avg_age"] = (
    df_players
    .groupby(["list_tournaments", "team_name"])["age"]
    .transform("mean")
)

In [103]:
df_players[df_players["team_name"]=="Argentina"].sort_values(by="list_tournaments").head()

,list_tournaments,team_name,position,age,avg_age
3983,1930,Argentina,Defender,24.0,24.863636
2115,1930,Argentina,Goalkeeper,25.0,24.863636
2126,1930,Argentina,Goalkeeper,22.0,24.863636
10297,1930,Argentina,Midfielder,29.0,24.863636
13795,1930,Argentina,Forward,22.0,24.863636


In [104]:
# Group by pos
df_players_age = (
    df_players
    .groupby(["list_tournaments", "team_name", "position"], as_index=False)
    .agg(
        avg_age_p_pos=("age", "mean"),
        avg_age=("avg_age", "first")
    )
).rename(columns={"list_tournaments": "year", "team_name": "team"})

In [105]:
df_players_age.head()

,year,team,position,avg_age_p_pos,avg_age
0,1930,Argentina,Defender,26.0,24.863636
1,1930,Argentina,Forward,23.8,24.863636
2,1930,Argentina,Goalkeeper,23.5,24.863636
3,1930,Argentina,Midfielder,26.4,24.863636
4,1930,Belgium,Defender,25.0,25.500000


### WC Standings per year 

Goal dataset: year, team, stage

In [106]:
df_standings = pd.read_csv("../data/raw/team_appearances_curated.csv")
df_standings.head()

,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,replay,...,goals_against,goal_differential,extra_time,penalty_shootout,penalties_for,penalties_against,result,win,lose,draw
0,eae9dec6-fcb0-5343-8f9b-c07e3895e50a,2007 FIFA Women's World Cup,0067a42b-9708-5c6f-bd60-0d20f08c0e3a,Germany vs Norway,semi-finals,NaN,False,True,False,False,...,0,3,False,False,NaN,NaN,win,True,False,False
1,eae9dec6-fcb0-5343-8f9b-c07e3895e50a,2007 FIFA Women's World Cup,0067a42b-9708-5c6f-bd60-0d20f08c0e3a,Germany vs Norway,semi-finals,NaN,False,True,False,False,...,3,-3,False,False,NaN,NaN,lose,False,True,False
2,c18a4889-5b51-5337-bc1a-a0731c1d97eb,2002 FIFA Men's World Cup,008ee188-ddec-5bd6-9ac8-b6ff21046bf8,Mexico vs Italy,group stage,Group G,True,False,False,False,...,1,0,False,False,NaN,NaN,draw,False,False,True
3,c18a4889-5b51-5337-bc1a-a0731c1d97eb,2002 FIFA Men's World Cup,008ee188-ddec-5bd6-9ac8-b6ff21046bf8,Mexico vs Italy,group stage,Group G,True,False,False,False,...,1,0,False,False,NaN,NaN,draw,False,False,True
4,09a48bd3-5401-5de4-9bd5-0e907ba0fee6,2015 FIFA Women's World Cup,00ac82e9-0086-5e5a-9408-7c100f6ff9f1,Sweden vs Nigeria,group stage,Group D,True,False,False,False,...,3,0,False,False,NaN,NaN,draw,False,False,True


In [107]:
# Filter only Men's WC
df_team_matches = (
    df_standings[
        df_standings["tournament_name"].str.contains(
            "FIFA Men's World Cup",
            na=False
        )
    ]
    .copy()
)

In [108]:
# Extract year
df_team_matches["year"] = (
    df_team_matches["tournament_name"]
    .str[:4]
    .astype(int)
)

In [109]:
sorted(df_team_matches["stage_name"].unique())

['final',
 'final round',
 'first group stage',
 'group stage',
 'quarter-finals',
 'round of 16',
 'second group stage',
 'semi-finals',
 'third-place match']

In [110]:
# Old WC had different formats, so a mapping has to be done
stage_rank = {
    "group stage": 1,
    "first group stage": 1,

    "second group stage": 2,
    "round of 16": 2,

    "quarter-finals": 3,

    "semi-finals": 4,
    "third-place match": 4,

    "final": 5,
    "final round": 5
}

df_team_matches["stage_score"] = (
    df_team_matches["stage_name"]
    .map(stage_rank)
)

In [111]:
# Obtain max stage reached
df_team_stage = (
    df_team_matches
    .groupby(
        ["year", "team_name"],
        as_index=False
    )
    .agg(
        stage_score=("stage_score", "max")
    )
    .rename(columns={
        "team_name": "team"
    })
)

In [112]:
df_team_stage.head(20)

,year,team,stage_score
0,1930,Argentina,5
1,1930,Belgium,1
2,1930,Bolivia,1
3,1930,Brazil,1
4,1930,Chile,1
5,1930,France,1
6,1930,Mexico,1
7,1930,Paraguay,1
8,1930,Peru,1
9,1930,Romania,1


In [113]:
df_team_stage["final_position"] = (
    df_team_stage["stage_score"]
    .map({
        1: "Group Stage",
        2: "Round of 16",
        3: "Quarter-finals",
        4: "Semi-finals",
        5: "Final"
    })
)

In [114]:
df_team_stage = df_team_stage[["year", "team", "stage_score", "final_position"]].copy()
df_team_stage

,year,team,stage_score,final_position
0,1930,Argentina,5,Final
1,1930,Belgium,1,Group Stage
2,1930,Bolivia,1,Group Stage
3,1930,Brazil,1,Group Stage
4,1930,Chile,1,Group Stage
...,...,...,...,...
484,2022,Switzerland,2,Round of 16
485,2022,Tunisia,1,Group Stage
486,2022,United States,2,Round of 16
487,2022,Uruguay,1,Group Stage


In [115]:
df_team_stage["stage_score"].unique()

array([5, 1, 4, 2, 3])

### Chart building

In [116]:
import plotly.express as px

In [117]:
# Merge datasets
df_age_stage = pd.merge(df_players_age, df_team_stage, on=["year", "team"])
df_avg_age = (
    df_age_stage[
        ["year", "team", "avg_age", "stage_score", "final_position"]
    ]
    .drop_duplicates(subset=["year", "team"])
)

In [118]:
df_avg_age.head()

,year,team,avg_age,stage_score,final_position
0,1930,Argentina,24.863636,5,Final
4,1930,Belgium,25.500000,1,Group Stage
8,1930,Bolivia,25.000000,1,Group Stage
12,1930,Brazil,24.791667,1,Group Stage
16,1930,Chile,26.368421,1,Group Stage


In [119]:
# Swarm Plot avg_age x final_position
order = (
    df_avg_age[["final_position", "stage_score"]]
    .drop_duplicates()
    .sort_values("stage_score")
    ["final_position"]
    .tolist()
)

fig = px.strip(
    df_avg_age, 
    x="final_position", 
    y="avg_age", 
    stripmode="group", 
    hover_data=["team", "year"], 
    category_orders={"final_position": order})
fig.show()

In [120]:
# Boxplot 
fig = px.box(
    df_avg_age, x="final_position", y="avg_age", 
    category_orders={"final_position": order})
fig.show()

In [121]:
# Heatmap
heatmap_data = df_age_stage.pivot_table(
    index='position',           # Rows: Tactical positions
    columns='final_position',   # Columns: Stages reached
    values='avg_age_p_pos',     # Heatmap cell values: Average age
    aggfunc='mean'              # Aggregate by taking the mean
)

position_order = ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']
stage_order = [
        "Group Stage",
        "Round of 16",
        "Quarter-finals",
        "Semi-finals",
        "Final"
]

# Apply the logical sorting, keeping only columns/rows that exist to prevent errors
existing_positions = [p for p in position_order if p in heatmap_data.index]
existing_stages = [s for s in stage_order if s in heatmap_data.columns]
heatmap_data = heatmap_data.loc[existing_positions, existing_stages]

# 3. Render the Heatmap with Plotly Express
fig_heatmap = px.imshow(
    heatmap_data,
    labels=dict(
        x="Tournament Stage", 
        y="Tactical Position", 
        color="Average Age"
    ),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    text_auto='.1f',                  # Display the age inside each cell with 1 decimal
    aspect="auto",                    # Stretch to fit the Streamlit container
    color_continuous_scale='YlOrRd',  # Yellow (young) to Red (old) color scale
    origin='upper'                    # Keeps Goalkeeper at the top of the Y axis
)

# Optional: Tweak the layout for a cleaner look
fig_heatmap.update_layout(
    xaxis_title=None,
    yaxis_title=None,
    plot_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=0, r=0, t=30, b=0)
)

fig_heatmap.show()

In [122]:
# Histogram
# User selects year and team

# Age distribution of all players
# Age distribution of selected team players
team_selected = "Argentina"
year_selected = 2022

df_hist = df_players[
    ["list_tournaments", "team_name", "age"]
].copy()

df_hist["group"] = "All Players"

df_team = df_hist[
    (df_hist["team_name"] == team_selected)
    & (df_hist["list_tournaments"] == year_selected)
].copy()

df_team["group"] = f"{team_selected} {year_selected}"

df_plot = pd.concat([df_hist, df_team], ignore_index=True)

fig = px.histogram(
    df_plot,
    x="age",
    color="group",
    barmode="overlay",
    histnorm="percent"
)

fig.update_traces(opacity=0.6)
fig.show()

### KPIs

In [123]:
# Finalists Average Age
df_age_stage = df_age_stage.drop_duplicates(subset=["year", "team"])
final_avg_age = df_age_stage[df_age_stage["final_position"]=="Final"]
final_avg_age = round(final_avg_age["avg_age"].mean())
print("Finalists Average Age:", final_avg_age)

# Group Stage Eliminated Average Age
group_avg_age = df_age_stage[df_age_stage["final_position"]=="Group Stage"]
group_avg_age = round(group_avg_age["avg_age"].mean())
print("Group Stage Eliminated Average Age:", group_avg_age)

# Gap between finalists and eliminated in group stage
gap = final_avg_age - group_avg_age
print("Gap between finalists and eliminated in group stage:", gap)

# Youngest and Oldest Finalists
df_finalists = df_age_stage[df_age_stage["final_position"].isin(["Final"])]
youngest = df_finalists.loc[df_finalists["avg_age"].idxmin()]
oldest = df_finalists.loc[df_finalists["avg_age"].idxmax()]

print("Youngest Finalist:", youngest["team"], youngest["year"],"->", round(youngest["avg_age"]), "years")
print("Oldest Finalist:", oldest["team"], oldest["year"],"->", round(oldest["avg_age"]), "years")


Finalists Average Age: 27
Group Stage Eliminated Average Age: 27
Gap between finalists and eliminated in group stage: 0
Youngest Finalist: Argentina 1930 -> 25 years
Oldest Finalist: Sweden 1958 -> 30 years


In [124]:
# Youngest and Oldest Champions
df_winners = pd.read_csv("../data/processed/winners.csv")
df_winners_age = df_winners.merge(df_finalists, on=["year","team"])
df_winners_age = df_winners_age[["year","team","position_x", "avg_age"]]
df_winners_age.head()

,year,team,position_x,avg_age
0,1930,Uruguay,1,26.681818
1,1930,Argentina,2,24.863636
2,1966,England,1,27.227273
3,1966,West Germany,2,25.590909
4,1978,Argentina,1,26.272727


In [125]:
champions = df_winners_age[df_winners_age["position_x"] == 1]
youngest = champions.loc[champions["avg_age"].idxmin()]
oldest = champions.loc[champions["avg_age"].idxmax()]

print(f"Youngest Champion: {youngest['team']} {youngest['year']} -> {youngest['avg_age']:.1f} years")
print(f"Oldest Champion: {oldest['team']} {oldest['year']} -> {oldest['avg_age']:.1f} years")

Youngest Champion: Brazil 1970 -> 25.0 years
Oldest Champion: Italy 2006 -> 28.8 years


## 2. Continental Hegemony

### Data Needed:
- WC Standings per Year (done)
- Team Confederation (UEFA, CONMEBOL, etc.)
- Matches Results
- Winners

### Team Confederation

Goal Dataset: team, confederation

In [126]:
df_confederation = pd.read_csv("../data/raw/teams_curated.csv")

In [127]:
df_confederation.head()

,team_id,team_name,team_code,men,women,federation_name,region_name,confederation_id,confederation_name,confederation_code,wikipedia_link_men,wikipedia_link_women,wikipedia_link_federation
0,0128c8d4-c488-5312-9fca-94d8f504a7a7,Bosnia and Herzegovina,BIH,True,False,Football Association of Bosnia and Herzegovina,Europe,e7c3477d-bcfb-5840-9489-cd082ab0954f,Union of European Football Associations,UEFA,https://en.wikipedia.org/wiki/Bosnia_and_Herze...,NaN,https://en.wikipedia.org/wiki/Football_Associa...
1,017c48cb-b97e-5de7-b7cd-7df9586d9d34,Trinidad and Tobago,TTO,True,False,Trinidad and Tobago Football Assocaition,Caribbean,2e2e8de6-629d-52db-ae57-a59213b1e0ed,"Confederation of North, Central American and C...",CONCACAF,https://en.wikipedia.org/wiki/Trinidad_and_Tob...,NaN,https://en.wikipedia.org/wiki/Trinidad_and_Tob...
2,0483a910-5880-55a5-abb2-5da0215a7d70,Iraq,IRQ,True,False,Iraq Football Association,Middle East,6349fd9d-fd40-5a60-9e90-8ff953842334,Asian Football Confederation,AFC,https://en.wikipedia.org/wiki/Iraq_national_fo...,NaN,https://en.wikipedia.org/wiki/Iraq_Football_As...
3,0db8bc49-3546-517c-af2c-7abd04ae2214,Croatia,HRV,True,False,Croatian Football Federation,Europe,e7c3477d-bcfb-5840-9489-cd082ab0954f,Union of European Football Associations,UEFA,https://en.wikipedia.org/wiki/Croatia_national...,NaN,https://en.wikipedia.org/wiki/Croatian_Footbal...
4,10913c23-6023-563d-94df-9b4fb66c3432,United States,USA,True,True,United States Soccer Federation,North America,2e2e8de6-629d-52db-ae57-a59213b1e0ed,"Confederation of North, Central American and C...",CONCACAF,https://en.wikipedia.org/wiki/United_States_me...,https://en.wikipedia.org/wiki/United_States_wo...,https://en.wikipedia.org/wiki/United_States_So...


In [128]:
df_confederation = df_confederation[["team_name", "confederation_code"]].copy()
df_confederation = df_confederation.rename(columns={"team_name": "team", "confederation_code": "confederation"})

In [129]:
df_confederation.head()

,team,confederation
0,Bosnia and Herzegovina,UEFA
1,Trinidad and Tobago,CONCACAF
2,Iraq,AFC
3,Croatia,UEFA
4,United States,CONCACAF


In [130]:
df_confederation["confederation"].unique()

<ArrowStringArray>
['UEFA', 'CONCACAF', 'AFC', 'OFC', 'CAF', 'CONMEBOL']
Length: 6, dtype: str

### Matches Results

Goal dataset: year, team, opponent, confederation, opponent_confederation, result

In [131]:
#Start: year, conf_home, conf_away, score_home, score_away
#End: year, team, opponent, confederation, opponent_confederation, result
df_matches = pd.read_csv("../data/raw/matches_curated.csv")

In [132]:
df_matches = (
    df_matches[
        df_matches["tournament_name"].str.contains(
            "FIFA Men's World Cup",
            na=False
        )
    ]
    .copy()
)
df_matches = df_matches[["match_date", "home_team_name", "away_team_name", "home_team_win", "away_team_win", "draw"]].copy()
df_matches.head()

,match_date,home_team_name,away_team_name,home_team_win,away_team_win,draw
1,2002-06-13,Mexico,Italy,False,False,True
3,1958-06-29,Brazil,Sweden,True,False,False
5,2022-12-06,Portugal,Switzerland,True,False,False
6,1950-07-16,Uruguay,Brazil,True,False,False
7,1938-06-05,Czechoslovakia,Netherlands,True,False,False


In [133]:
# Add result
df_matches["result"] = np.select(
    [
        df_matches["home_team_win"],
        df_matches["away_team_win"]
    ],
    [
        "Win",
        "Loss"
    ],
    default="Draw"
)

# Date to Year
df_matches["year"] = (
    df_matches["match_date"]
    .str[:4]
    .astype(int)
)
df_matches = df_matches[["year", "home_team_name", "away_team_name", "result"]]
df_matches.head()


,year,home_team_name,away_team_name,result
1,2002,Mexico,Italy,Draw
3,1958,Brazil,Sweden,Win
5,2022,Portugal,Switzerland,Win
6,1950,Uruguay,Brazil,Win
7,1938,Czechoslovakia,Netherlands,Win


In [134]:
# Add team confederation
conf_map = df_confederation.set_index("team")["confederation"]

df_matches["team_conf"] = df_matches["home_team_name"].map(conf_map)
df_matches["opponent_conf"] = df_matches["away_team_name"].map(conf_map)

In [135]:
df_matches.rename(columns={"home_team_name": "team", "away_team_name": "opponent_team"}, inplace=True)
df_matches.head()

,year,team,opponent_team,result,team_conf,opponent_conf
1,2002,Mexico,Italy,Draw,CONCACAF,UEFA
3,1958,Brazil,Sweden,Win,CONMEBOL,UEFA
5,2022,Portugal,Switzerland,Win,UEFA,UEFA
6,1950,Uruguay,Brazil,Win,CONMEBOL,CONMEBOL
7,1938,Czechoslovakia,Netherlands,Win,UEFA,UEFA


In [136]:
# Add another row per match but for the opponent_team
home = df_matches.copy()

away = df_matches.copy()

away["team"] = df_matches["opponent_team"]
away["opponent_team"] = df_matches["team"]

away["team_conf"] = df_matches["opponent_conf"]
away["opponent_conf"] = df_matches["team_conf"]

away["result"] = away["result"].map({
    "Win": "Loss",
    "Loss": "Win",
    "Draw": "Draw"
})

df_matches_long = pd.concat(
    [home, away],
    ignore_index=True
)


In [137]:
# For example:
df_matches_long[(df_matches_long["team"]=="Mexico") & (df_matches_long["opponent_team"]=="Sweden") | (df_matches_long["team"]=="Sweden") & (df_matches_long["opponent_team"]=="Mexico")]

,year,team,opponent_team,result,team_conf,opponent_conf
49,2018,Mexico,Sweden,Loss,CONCACAF,UEFA
648,1958,Sweden,Mexico,Win,UEFA,CONCACAF
1013,2018,Sweden,Mexico,Win,UEFA,CONCACAF
1612,1958,Mexico,Sweden,Loss,CONCACAF,UEFA


### Winners per Year

In [138]:
winners = pd.read_csv("../data/raw/tournament_standings_curated.csv")
winners = winners[winners["tournament_name"].str.contains("FIFA Men's World Cup",)].copy()
winners["year"] = winners["tournament_name"].str[0:4]
winners = winners[["year", "team_name", "position"]]
winners = winners.rename(columns={"team_name": "team"})

In [139]:
winners.head()

,year,team,position
12,1930,Uruguay,1
13,1930,Argentina,2
14,1930,United States,3
15,1930,Yugoslavia,4
16,1966,England,1


### Chart Building

In [140]:
df_standings = pd.read_csv("../data/processed/standings.csv")

In [141]:
# Merge datasets
df_conf_standings = pd.merge(df_standings, df_confederation, on=["team"])

In [142]:
df_conf_standings.head()

,year,team,stage_score,final_position,confederation
0,1930,Argentina,5,Final,CONMEBOL
1,1930,Belgium,1,Group Stage,UEFA
2,1930,Bolivia,1,Group Stage,CONMEBOL
3,1930,Brazil,1,Group Stage,CONMEBOL
4,1930,Chile,1,Group Stage,CONMEBOL


In [143]:
# Scatter Rest of the World (Knockout stages)
order = (
    df_conf_standings[["final_position", "stage_score"]]
    .drop_duplicates()
    .sort_values("stage_score")
    ["final_position"]
    .tolist()
) 

rest_of_world = df_conf_standings[
    (~df_conf_standings["confederation"].isin(["UEFA", "CONMEBOL"])) &
    (df_conf_standings["stage_score"] > 1)
]

fig = px.scatter(
    rest_of_world,
    x="year",
    y="final_position",
    color="confederation",
    category_orders={"final_position": order},
    hover_data=["team"]
)
fig.update_yaxes(autorange="reversed")
fig.show()

In [144]:
# Stacked Bar Chart Final Positions
order = (
    df_conf_standings[["final_position", "stage_score"]]
    .drop_duplicates()
    .sort_values("stage_score")
    ["final_position"]
    .tolist()
) 

fig = px.histogram(
    df_conf_standings,
    x="final_position",
    color="confederation",
    barmode="stack",
    category_orders={"final_position": order}
)

fig.show()

In [145]:
# Heatmap
# Exclude OFC to avoid statistic noise due to the low amount of matches
df_matches = df_matches[
    (df_matches['team_conf'] != 'OFC') & 
    (df_matches['opponent_conf'] != 'OFC')
]

# Win = 1, resto = 0
df_matches["win"] = (
    df_matches["result"] == "Win"
).astype(int)

# Only UEFA y CONMEBOL agains the rest
df_heatmap = df_matches[
    (df_matches["team_conf"].isin(["UEFA", "CONMEBOL"])) &
    (df_matches["opponent_conf"].isin(["CAF", "CONCACAF", "AFC", "OFC"]))
]

# Win rate
win_rate = (
    df_heatmap
    .groupby(["team_conf", "opponent_conf"])["win"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)

# Matrix
heatmap = win_rate.pivot(
    index="team_conf",
    columns="opponent_conf",
    values="win"
)

# Order
heatmap = heatmap.reindex(
    index=["UEFA", "CONMEBOL"],
    columns=["CAF", "CONCACAF", "AFC"]
)

# Chart
fig = px.imshow(
    heatmap,
    text_auto=".1f",
    color_continuous_scale="Greens",
    aspect="auto",
    zmin=0,
    zmax=100,
    labels={
        "x": "Opponent Confederation",
        "y": "Confederation",
        "color": "Win Rate (%)"
    }
)

fig.update_layout(
    title="Win Rate vs Other Confederations",
    xaxis_title=None,
    yaxis_title=None,
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

In [146]:
# Sankey
"""rafico Complementario: Diagrama de Sankey (Flujo de Supervivencia): 
Este es un gráfico avanzado visualmente espectacular para Streamlit. 
A la izquierda tenés los 32 equipos divididos por confederación. 
De ahí salen "ríos" hacia la derecha que pasan por Grupos -> Octavos -> Cuartos. 
Visualmente vas a ver cómo el "río" de África o Asia se seca rápido, mientras que el de UEFA 
llega gordo hasta el final."""
import plotly.graph_objects as go

# Confederations
conf_map = df_confederation.set_index("team")["confederation"]

df_standings["confederation"] = (
    df_standings["team"]
    .map(conf_map)
)

# Group rest
df_standings["conf_group"] = (
    df_standings["confederation"]
    .replace({
        "CAF": "Rest of the World",
        "AFC": "Rest of the World",
        "CONCACAF": "Rest of the World",
        "OFC": "Rest of the World"
    })
)

# Survive
survival = (
    df_standings
    .groupby("conf_group")
    .agg(
        group_stage=("team", "count"),
        round16=("stage_score", lambda x: (x >= 2).sum()),
        quarterfinals=("stage_score", lambda x: (x >= 3).sum()),
        semifinals=("stage_score", lambda x: (x >= 4).sum()),
        finals=("stage_score", lambda x: (x >= 5).sum()),
    )
)

# Order
survival = survival.loc[
    ["UEFA", "CONMEBOL", "Rest of the World"]
]

# Nodes
labels = [
    "UEFA GS", "UEFA R16", "UEFA QF", "UEFA SF", "UEFA Final",
    "CONMEBOL GS", "CONMEBOL R16", "CONMEBOL QF", "CONMEBOL SF", "CONMEBOL Final",
    "Rest of the World GS", "Rest of the World R16", "Rest of the World QF", "Rest of the World SF", "Rest of the World Final",
]

# Colors
conf_colors = {
    "UEFA": "rgba(54, 162, 235, 0.45)",
    "CONMEBOL": "rgba(46, 204, 113, 0.45)",
    "Rest of the World": "rgba(160, 160, 160, 0.35)"
}

node_colors = (
    ["#36A2EB"] * 5 +
    ["#2ECC71"] * 5 +
    ["#A0A0A0"] * 5
)

source = []
target = []
value = []
link_colors = []

conf_offsets = {
    "UEFA": 0,
    "CONMEBOL": 5,
    "Rest of the World": 10
}

for conf in ["UEFA", "CONMEBOL", "Rest of the World"]:

    offset = conf_offsets[conf]

    vals = [
        survival.loc[conf, "round16"],
        survival.loc[conf, "quarterfinals"],
        survival.loc[conf, "semifinals"],
        survival.loc[conf, "finals"],
    ]

    for i, v in enumerate(vals):
        source.append(offset + i)
        target.append(offset + i + 1)
        value.append(v)
        link_colors.append(conf_colors[conf])

fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=labels,
            color=node_colors,
            pad=20,
            thickness=18,
            line=dict(color="black", width=0.5)
        ),
        link=dict(
            source=source,
            target=target,
            value=value,
            color=link_colors
        )
    )
)

fig.update_layout(
    title="World Cup Survival by Confederation",
    font_size=12,
    height=700
)

fig.show()

### KPIs

In [147]:
df_conf_standings.head()

,year,team,stage_score,final_position,confederation
0,1930,Argentina,5,Final,CONMEBOL
1,1930,Belgium,1,Group Stage,UEFA
2,1930,Bolivia,1,Group Stage,CONMEBOL
3,1930,Brazil,1,Group Stage,CONMEBOL
4,1930,Chile,1,Group Stage,CONMEBOL


In [148]:
# UEFA and CONMEBOL teams % in Semifinals
semis = df_conf_standings[df_conf_standings["stage_score"] >= 4]
pct = (
    semis["confederation"]
    .isin(["UEFA", "CONMEBOL"])
    .mean()
    * 100
)

print(f"Historic Semifinalists from UEFA/CONMEBOL: {pct:.1f}%")

Historic Semifinalists from UEFA/CONMEBOL: 96.6%


In [149]:
# Exceptions: Semifinal Appearances Outside UEFA/CONMEBOL
exceptions = df_conf_standings[
    (df_conf_standings["stage_score"] >= 4) &
    (~df_conf_standings["confederation"].isin(["UEFA", "CONMEBOL"]))
]

exc_count = len(exceptions)

print("Semifinal Appearances Outside UEFA/CONMEBOL:", exc_count)

Semifinal Appearances Outside UEFA/CONMEBOL: 3


In [150]:
# Survival rate "Rest of the World": of Asia/Africa/Concacaf what % survives a group fase.

rest_of_world = df_conf_standings[
    ~df_conf_standings["confederation"].isin(["UEFA", "CONMEBOL"])
]

survive_group = rest_of_world[
    rest_of_world["stage_score"] > 1
]

survival_rate = (
    len(survive_group)
    / len(rest_of_world)
    * 100
)

print(f"Group Stage Survival Rate (Non-UEFA/CONMEBOL): {survival_rate:.1f}%")


Group Stage Survival Rate (Non-UEFA/CONMEBOL): 30.5%


In [151]:
# WC Titles per Confederation
df_conf_winners = pd.merge(winners, df_confederation, on=["team"])

wc_titles = df_conf_winners[
    df_conf_winners["position"] == 1
]

uefa_wc_titles = (
    wc_titles["confederation"]
    .eq("UEFA")
    .sum()
)

conmebol_wc_titles = (
    wc_titles["confederation"]
    .eq("CONMEBOL")
    .sum()
)

rest_wc_titles = (
    ~wc_titles["confederation"]
    .isin(["UEFA", "CONMEBOL"])
).sum()

print("UEFA:", uefa_wc_titles)
print("CONMEBOL:", conmebol_wc_titles)
print("REST:", rest_wc_titles)


UEFA: 12
CONMEBOL: 10
REST: 0


In [152]:
winners

,year,team,position
12,1930,Uruguay,1
13,1930,Argentina,2
14,1930,United States,3
15,1930,Yugoslavia,4
16,1966,England,1
...,...,...,...
119,1962,Yugoslavia,4
120,1998,France,1
121,1998,Brazil,2
122,1998,Croatia,3


## 3. The First Kick Advantage

### Data Needed:
- WC Penalty Shootouts Historical Record (From Scraping)

In [153]:
df_penalty = pd.read_csv("../data/processed/penalty_shootouts.csv")

In [154]:
df_penalty.head(10)

,year,round,winner,loser,team,taker,penalty_order,scored,first_taker
0,1982,Semi-finals,West Germany,France,West Germany,Kaltz,1,True,False
1,1982,Semi-finals,West Germany,France,West Germany,Breitner,2,True,False
2,1982,Semi-finals,West Germany,France,West Germany,Stielike,3,False,False
3,1982,Semi-finals,West Germany,France,West Germany,Littbarski,4,True,False
4,1982,Semi-finals,West Germany,France,West Germany,Rummenigge,5,True,False
5,1982,Semi-finals,West Germany,France,France,Giresse,1,True,True
6,1982,Semi-finals,West Germany,France,France,Amoros,2,True,False
7,1982,Semi-finals,West Germany,France,France,Rocheteau,3,True,False
8,1982,Semi-finals,West Germany,France,France,Six,4,False,False
9,1982,Semi-finals,West Germany,France,France,Platini,5,True,False


### Chart Building

In [155]:
# Tornado chart
# To the left: conversion rate from the team that shooted first
# To the right: conversion rate from the team that shooted second

df = df_penalty.copy()

# Create id per match
df['match_id'] = df['year'].astype(str) + "_" + df['winner'] + "_" + df['loser']

first_kicking_teams = df[df['first_taker'] == True][['match_id', 'team']].drop_duplicates()
first_kicking_teams['kicked_first'] = True

df = df.merge(first_kicking_teams, on=['match_id', 'team'], how='left')
df['kicked_first'] = df['kicked_first'].fillna(False)

df_filtered = df[df['penalty_order'] <= 5]

conversion_rates = df_filtered.groupby(['penalty_order', 'kicked_first'])['scored'].mean().reset_index()
conversion_rates['conversion_pct'] = (conversion_rates['scored'] * 100).round(1)

team1_data = conversion_rates[conversion_rates['kicked_first'] == True].sort_values('penalty_order')
team2_data = conversion_rates[conversion_rates['kicked_first'] == False].sort_values('penalty_order')

# Chart
y_labels = [f"Ronda {i}" for i in range(1, 6)]
fig_tornado = go.Figure()

# Left bars
fig_tornado.add_trace(go.Bar(
    y=y_labels,
    x=-team1_data['conversion_pct'],
    name='Pateó Primero',
    orientation='h',
    marker_color='#2ecc71', # Verde
    text=team1_data['conversion_pct'].astype(str) + '%',
    textposition='inside',
    hoverinfo='text',
    hovertext='Ronda ' + team1_data['penalty_order'].astype(str) + '<br>Conversión: ' + team1_data['conversion_pct'].astype(str) + '%'
))

# Right bars
fig_tornado.add_trace(go.Bar(
    y=y_labels,
    x=team2_data['conversion_pct'],
    name='Pateó Segundo',
    orientation='h',
    marker_color='#e74c3c', # Rojo
    text=team2_data['conversion_pct'].astype(str) + '%',
    textposition='inside',
    hoverinfo='text',
    hovertext='Ronda ' + team2_data['penalty_order'].astype(str) + '<br>Conversión: ' + team2_data['conversion_pct'].astype(str) + '%'
))

# Show negative numbers as positives
fig_tornado.update_layout(
    barmode='overlay',
    height=500,
    width=800, 
    xaxis=dict(
        tickvals=[-100, -75, -50, -25, 0, 25, 50, 75, 100],
        ticktext=['100%', '75%', '50%', '25%', '0', '25%', '50%', '75%', '100%'],
        range=[-110, 110],
        title='Tasa de Conversión (%)'
    ),
    yaxis=dict(autorange="reversed"),
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    plot_bgcolor='white'
)

# Mostrar en Jupyter Notebook
fig_tornado.show()

In [156]:
# Bar: scoring first penalty gives higher chance to win 
first_scored_win = df_penalty[(df_penalty["penalty_order"]==1) & (df_penalty["scored"]==True) & (df_penalty["winner"]==df_penalty["team"])].shape[0]
first_missed_win = df_penalty[(df_penalty["penalty_order"]==1) & (df_penalty["scored"]==False) & (df_penalty["winner"]==df_penalty["team"])].shape[0]
first_scored_total = df_penalty[(df_penalty["penalty_order"]==1) & (df_penalty["scored"]==True)].shape[0]
first_missed_total = df_penalty[(df_penalty["penalty_order"]==1) & (df_penalty["scored"]==False)].shape[0]

first_scored_win_rate = first_scored_win / first_scored_total * 100
first_missed_win_rate = first_missed_win / first_missed_total * 100

df_chart = pd.DataFrame({
    "First penalty": [
        "Scored",
        "Missed",
    ],
    "Win rate": [
        first_scored_win / first_scored_total * 100,
        first_missed_win / first_missed_total * 100,
    ],
})

fig = px.bar(
    df_chart,
    x="First penalty",
    y="Win rate",
    text="Win rate",
)

fig.update_traces(
    texttemplate="%{y:.1f}%",
    textposition="outside",
)

fig.update_yaxes(
    title="Win rate (%)",
    range=[0, 100],
)

fig.update_layout(
    title="Win Probability Based on First Penalty Outcome",
    showlegend=False,
)

fig.show()

In [157]:
max_penalty = (
    df_penalty.groupby(["year", "round", "winner", "loser"])
    .agg(
        max_penalty=("penalty_order", "max")
    )
      .reset_index()
)

ended_before = max_penalty[max_penalty["max_penalty"]<5].shape[0]
reached_fifth = max_penalty[max_penalty["max_penalty"]==5].shape[0]
total = ended_before + reached_fifth

df_chart = pd.DataFrame({
    "Category": [
        "Ended before 5th kick",
        "Reached 5th kick",
    ],
    "Count": [
        ended_before,
        reached_fifth,
    ]
})

df_chart["Percentage"] = df_chart["Count"] / total * 100

fig = px.bar(
    df_chart,
    x="Percentage",
    y=["Shootouts"] * len(df_chart),   # una sola barra
    color="Category",
    orientation="h",
    text=df_chart["Percentage"].round(1).astype(str) + "%",
)

fig.update_traces(
    textposition="inside",
)

fig.update_layout(
    barmode="stack",
    showlegend=True,
    xaxis_title="Percentage of shootouts",
    yaxis_title="",
)

fig.update_xaxes(range=[0, 100])

fig.show()

### KPIs

In [158]:
# First team to take penalty win rate (%)
# One row per shootout
shootouts = (
    df_penalty.groupby(["year", "round", "winner", "loser"])
      .first()
      .reset_index()
)

first_team = (
    df_penalty[df_penalty["first_taker"]]
    .groupby(["year", "round", "winner", "loser"])["team"]
    .first()
    .reset_index(name="first_team")
)

shootouts = shootouts.merge(
    first_team,
    on=["year", "round", "winner", "loser"]
)

shootouts["first_team_won"] = (
    shootouts["first_team"] == shootouts["winner"]
)

win_rate = shootouts["first_team_won"].mean() * 100

print(f"First team to take penalty win rate: {win_rate:.0f}%")



First team to take penalty win rate: 49%


In [159]:
# Total penalty shootouts
n_shootouts = (
    df_penalty[["year", "round", "winner", "loser"]]
    .drop_duplicates()
    .shape[0]
)

print(n_shootouts)

35


In [160]:
# Penalties scored %
penalties_scored = (df_penalty[df_penalty["scored"]==True].shape[0] / df_penalty.shape[0])*100

print(f"{penalties_scored:.0f}%")

69%


## Others

In [161]:
# Last WC
last_wc = pd.read_csv("../data/processed/winners.csv")
last_wc_nr = last_wc.nlargest(1, "year")["year"].iloc[0]

print(f"Last WC in dataset: {last_wc_nr}")

Last WC in dataset: 2022
